# 🎬 SnapFoley — Cloud Demo (Google Colab, free)

Runs the **whole SnapFoley backend in the cloud** and gives you a **public URL** you can open from any device.

**How to use:**
1. Menu: `Runtime` → `Run all` (or run each cell top-to-bottom)
2. Wait for the last cell to print **PUBLIC URL**
3. Open that URL in any browser → upload your video → process → generate final video with audio

Notes:
- First run installs dependencies (~3–5 min)
- Colab sessions last a few hours and die if idle — just re-run all cells for a fresh URL
- If Colab gives you a T4 GPU (`Runtime` → `Change runtime type`), detection runs even faster automatically
- 100% offline pipeline: no third-party APIs, no keys

In [ ]:
# 1) Download the SnapFoley code from the free HF dataset repo + install deps (~3 min)
from huggingface_hub import snapshot_download
import os, sys

SRV_DIR = "/content/snapfoley"
snapshot_download(repo_id="ritohere/snapfoley-code", repo_type="dataset", local_dir=SRV_DIR)
os.chdir(SRV_DIR)
print("code at:", SRV_DIR)
print("python:", sys.version.split()[0])
!pip install -q -r requirements-colab.txt
!ffmpeg -version 2>/dev/null | head -n 1 || echo "ffmpeg missing!"

In [ ]:
# 2) Start the backend server (background) and wait for /api/health
import sys, subprocess, time, requests
SRV_DIR = "/content/snapfoley"

subprocess.run(["pkill", "-f", "uvicorn"], capture_output=True)  # clean restart
log = open(f"{SRV_DIR}/server.log", "w")
srv = subprocess.Popen(
    [sys.executable, "-m", "uvicorn", "app.main:app", "--host", "0.0.0.0", "--port", "7860"],
    cwd=SRV_DIR, stdout=log, stderr=subprocess.STDOUT,
)
print("uvicorn starting (pid", srv.pid, ")...")
for _ in range(90):
    time.sleep(2)
    try:
        r = requests.get("http://127.0.0.1:7860/api/health", timeout=2)
        if r.ok:
            print("SERVER UP:", r.json())
            break
    except Exception:
        pass
else:
    print("server did NOT come up — last log lines:")
    print(open(f"{SRV_DIR}/server.log").read()[-3000:])

In [ ]:
# 3) Public URL via Cloudflare tunnel (no account needed)
import subprocess, time, re, os
CF = "/content/cloudflared"

subprocess.run(["pkill", "-f", "cloudflared"], capture_output=True)  # clean restart
if not os.path.exists(CF):
    subprocess.run(["wget", "-q",
                    "https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64",
                    "-O", CF], check=True)
    os.chmod(CF, 0o755)
cf = subprocess.Popen([CF, "tunnel", "--url", "http://127.0.0.1:7860", "--no-autoupdate",
                       "--logfile", "/content/cf.log"],
                      stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
url = None
for _ in range(45):
    time.sleep(2)
    try:
        txt = open("/content/cf.log").read()
    except FileNotFoundError:
        continue
    m = re.findall(r"https://[a-z0-9-]+\.trycloudflare\.com", txt)
    if m:
        url = m[-1]
        break
print("=" * 60)
print("  PUBLIC URL:", url)
print("=" * 60)
print("Open that URL in any browser — the SnapFoley web UI is served at /")
print("(URL is unique to this session; re-run all cells after a restart)")

In [ ]:
# 3b) STABLE public URL via ngrok free static domain (the link NEVER changes)
# One-time setup (2 min):
#   1. Sign up free: https://dashboard.ngrok.com/signup
#   2. Claim your free static domain: https://dashboard.ngrok.com/domains -> New domain -> Save
#      (ngrok suggests a random name like 'xyz-wxy.ngrok-free.app' — you can pick a custom one if free)
#   3. In Colab click the KEY icon (left sidebar) -> Add secret (turn ON Notebook access):
#        NGROK_TOKEN  = your authtoken from https://dashboard.ngrok.com/get-started/your-authtoken
#        NGROK_DOMAIN = your static domain from step 2  (e.g. snapfoley-demo.ngrok-free.app)
#   4. Run this cell — the same URL works in every future session.
from google.colab import userdata
def _sec(name):
    try:
        return userdata.get(name)
    except Exception:
        return ""
NGROK_TOKEN = _sec("NGROK_TOKEN")
NGROK_DOMAIN = _sec("NGROK_DOMAIN")
if NGROK_TOKEN and NGROK_DOMAIN:
    get_ipython().system('pip install -q pyngrok')
    from pyngrok import ngrok, conf
    conf.get_default().auth_token = NGROK_TOKEN
    pub = ngrok.connect(7860, domain=NGROK_DOMAIN)
    print("STABLE PUBLIC URL:", pub.public_url)
    print("(First visitors see a one-time ngrok 'Visit Site' interstitial — that is normal on free tier.)")
else:
    print("ngrok skipped — add Colab secrets NGROK_TOKEN + NGROK_DOMAIN (key icon) for a permanent link.")

In [ ]:
# 4) Check what the backend is doing anytime (upload/process logs)
!tail -n 30 /content/snapfoley/server.log

## Troubleshooting
- **"failed to fetch" in the web UI** → the server restarted; run cell 2 and 3 again
- **URL stopped working** → Colab session ended (idle timeout); `Runtime` → `Run all`
- **Install errors** → `Runtime` → `Disconnect and delete runtime`, then `Run all` again
- **Cloudflare error 1033** → that session ended; `Runtime` → `Run all` gives a fresh URL (set up cell 3b with ngrok for a link that never changes)
- Generated files live in `/content/snapfoley/tasks/` while the session lasts